# Gaia DR3: a million-star Hertzsprung–Russell diagram

This notebook queries the official ESA Gaia Archive and gives the raw
color–magnitude points to XY. The main sequence, red-giant branch, and
white-dwarf sequence emerge as density structure; no pre-binning is
required.

Gaia DR3 contains roughly 1.8 billion sources. The default query keeps
this run practical at one million high-quality stars. Increase
`GAIA_ROWS` to exercise a larger slice.

**Source:** [Gaia Archive programmatic access](https://www.cosmos.esa.int/web/gaia-users/archive/programmatic-access)
and [`gaiadr3.gaia_source`](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_tables/ssec_dm_gaia_source.html).

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROW_LIMIT = int(os.getenv("GAIA_ROWS", "1000000"))
if ROW_LIMIT <= 0:
    raise ValueError("GAIA_ROWS must be positive")

TAP_SYNC = "https://gea.esac.esa.int/tap-server/tap/sync"
query = f"""
SELECT TOP {ROW_LIMIT}
    bp_rp,
    phot_g_mean_mag,
    parallax
FROM gaiadr3.gaia_source
WHERE bp_rp IS NOT NULL
    AND phot_g_mean_mag IS NOT NULL
    AND parallax > 0
    AND parallax_over_error > 10
    AND phot_g_mean_flux_over_error > 50
"""

csv_path = DATA_DIR / f"gaia-dr3-hr-{ROW_LIMIT}.csv"
if not csv_path.exists():
    with requests.post(
        TAP_SYNC,
        data={
            "REQUEST": "doQuery",
            "LANG": "ADQL",
            "FORMAT": "csv",
            "QUERY": query,
        },
        stream=True,
        timeout=(30, 3600),
    ) as response:
        response.raise_for_status()
        partial = csv_path.with_suffix(".csv.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
        partial.replace(csv_path)

print(f"cached query result: {csv_path}")

In [ ]:
stars = np.genfromtxt(
    csv_path,
    delimiter=",",
    names=True,
    dtype=np.float64,
    encoding="utf-8",
)
stars = np.atleast_1d(stars)

color_index = stars["bp_rp"]
absolute_g = stars["phot_g_mean_mag"] + 5 * np.log10(stars["parallax"]) - 10
finite = np.isfinite(color_index) & np.isfinite(absolute_g)
color_index = color_index[finite]
absolute_g = absolute_g[finite]

print(f"{color_index.size:,} stars")

In [ ]:
axis_style = {
    "axis_color": "#65708d",
    "axis_width": 1.0,
    "grid_color": "#343755",
    "grid_dash": "dotted",
    "grid_opacity": 0.58,
    "grid_width": 0.8,
    "label_color": "#d7d9e8",
    "label_size": 14,
    "tick_color": "#65708d",
    "tick_label_color": "#aeb4cb",
    "tick_label_size": 12.5,
    "tick_length": 5,
    "tick_width": 0.8,
}
sequence_label_style = {
    "background": "#090b18f2",
    "border": "1px solid #8b83b880",
    "border_radius": 7,
    "font_size": 14.5,
    "font_weight": 700,
    "letter_spacing": "0.055em",
    "padding": "4px 8px",
}

chart = xy.scatter_chart(
    xy.scatter(
        color_index,
        absolute_g,
        color=absolute_g,
        color_domain=(-6.0, 16.0),
        colormap="magma_r",
        size=1.0,
        opacity=0.82,
        density=True,
    ),
    xy.callout(
        2.35,
        0.8,
        "RED GIANT BRANCH",
        dx=88,
        dy=-32,
        color="#f6a15f",
        width=1.2,
        opacity=0.8,
        style={**sequence_label_style, "label_color": "#ffc48d"},
    ),
    xy.callout(
        1.72,
        6.1,
        "MAIN SEQUENCE",
        dx=116,
        dy=-22,
        color="#bca8ff",
        width=1.2,
        opacity=0.76,
        style={**sequence_label_style, "label_color": "#ddd4ff"},
    ),
    xy.callout(
        0.65,
        9.0,
        "WHITE DWARFS",
        dx=-86,
        dy=34,
        color="#8ed8ff",
        width=1.2,
        opacity=0.78,
        anchor="end",
        style={**sequence_label_style, "label_color": "#bde9ff"},
    ),
    xy.text(
        0.08,
        0.965,
        "GAIA DR3",
        dx=0,
        dy=0,
        color="#bca8ff",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 11.5,
            "font_weight": 750,
            "letter_spacing": "0.18em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.08,
        0.928,
        "Hertzsprung\u2013Russell diagram",
        dx=0,
        dy=0,
        color="#fff5e8",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 25,
            "font_weight": 760,
            "letter_spacing": "0.005em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.08,
        0.872,
        f"{color_index.size:,} HIGH-CONFIDENCE STARS  ·  BRIGHTER ↑  ·  BLUE → RED",
        dx=0,
        dy=0,
        color="#9299b8",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 11.5,
            "font_weight": 600,
            "letter_spacing": "0.09em",
            "vertical_align": "top",
        },
    ),
    xy.x_axis(
        label="STELLAR COLOR  ·  BP \u2212 RP (mag)",
        domain=(-1.0, 5.0),
        tick_values=[-1, 0, 1, 2, 3, 4, 5],
        tick_labels=["\u22121", "0", "1", "2", "3", "4", "5"],
        style=axis_style,
    ),
    xy.y_axis(
        label="ABSOLUTE G MAGNITUDE",
        label_offset=-28,
        domain=(-6.0, 16.0),
        reverse=True,
        tick_values=[-6, -2, 2, 6, 10, 14],
        tick_labels=["\u22126", "\u22122", "2", "6", "10", "14"],
        style=axis_style,
    ),
    xy.theme(
        background="#03040c",
        plot_background="#080916",
        text_color="#e9eaf2",
        grid_color="#343755",
        axis_color="#65708d",
        crosshair_color="#ffc48d",
        selection_color="#bca8ff",
        selection_fill="#bca8ff26",
    ),
    styles={
        "annotation_label": {"line_height": 1.2},
        "axis_title": {"font_weight": 680, "letter_spacing": "0.055em"},
        "tick_label": {"font_variant_numeric": "tabular-nums"},
    },
    padding=(112, 48, 70, 116),
    width=1200,
    height=700,
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart